# HyperSIGMA native-geometry MAE SEM-tuning (Ablation 2)

Tune the **SEM fusion module** in the MAE way (like the existing `joint_sem` SEM-tuning runs), but on top
of the **frozen native-geometry encoder** (SpatViT 64x64/patch-8/100ch, SpecViT 64x64) with the 11x11 patch
**upscaled** to 64x64 and **PCA 144->100** spatial channels (Trento/MUUFL spectrally resample 63/64->100).

`adapt_mode="sem_only"`: the pretrained encoder is fully frozen; only the **fusion path** trains via the
fused-reconstruction loss:

* **trained**: SEM + fused decoder + the random spectral `l1` projection + the two mask tokens
* **frozen**: both pretrained branch bodies + `patch_embed`/`spat_map`/`pos_embed` + PCA front-end

For each dataset this notebook (1) ensures the 100-band PCA (Houston), (2) runs the SEM-tuning adaptation,
(3) evaluates the adapted **fused** features few-shot. Existing notebooks/configs are untouched.

> Per project convention the GPU adapt + eval cells are run by **you**.


In [2]:
# === Parameters ===
import os, sys
from pathlib import Path as _Path

_REPO = _Path.cwd()
while not (_REPO / "lib" / "experiments.py").exists():
    if _REPO.parent == _REPO:
        raise RuntimeError("Could not locate repo root containing lib/experiments.py")
    _REPO = _REPO.parent
os.chdir(_REPO)
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

from data.datasets import get_spec

# --------------------------------------------------------------------------
datasets = ["houston", "trento", "muufl"]   # all three (subset by editing)
input_fit = "upscale"                        # the user-requested 11->64 fit

# Base config (Houston native-SEM); per-dataset bits are deep-merged below.
adapt_config_path = "configs/pretrain/hypersigma_houston_adapt_native_sem.yaml"

# Few-shot eval episode shape (C-way; matches the other HyperSIGMA notebooks).
k_shot, k_query, num_episodes = 5, 100, 2000

# Umbrella experiment for the fused evals; per-dataset adapt experiments below.
experiment_name = "hypersigma_native_sem_run1"

adapt_device = "cuda:2"     # adaptation GPU
eval_device  = "cuda:2"     # evaluation GPU
adapt_epochs = 2000         # SEM-only trains few params; tune as needed
overwrite = True

NATIVE_SPAT_IN_CHANS = 100
PCA_DIR = _Path("checkpoints/hypersigma")

# Per-dataset native spatial front-end: Houston (>=100 bands) -> PCA 144->100;
# Trento/MUUFL (<100 bands) -> spectral resample (native_pca_spat_path = None).
def native_pca_path(ds):
    spec = get_spec(ds)
    if spec.hsi_channels >= NATIVE_SPAT_IN_CHANS:
        return f"checkpoints/hypersigma/pca_{ds}_{NATIVE_SPAT_IN_CHANS}band.pkl"
    return None

for ds in datasets:
    s = get_spec(ds)
    print(f"  {ds:8s} bands={s.hsi_channels:3d} -> spatial 100ch via "
          f"{'PCA 144->100' if s.hsi_channels >= NATIVE_SPAT_IN_CHANS else f'resample {s.hsi_channels}->100'}")
print(f"\nadapt_mode=sem_only, native_geometry=True, input_fit={input_fit}")
print(f"experiment_name = {experiment_name}")


  houston  bands=144 -> spatial 100ch via PCA 144->100
  trento   bands= 63 -> spatial 100ch via resample 63->100
  muufl    bands= 64 -> spatial 100ch via resample 64->100

adapt_mode=sem_only, native_geometry=True, input_fit=upscale
experiment_name = hypersigma_native_sem_run1


In [3]:
# Repo root + autoreload + torch info.
import os, sys
from pathlib import Path
REPO = Path.cwd()
while not (REPO / "lib" / "experiments.py").exists():
    REPO = REPO.parent
os.chdir(REPO); sys.path.insert(0, str(REPO)) if str(REPO) not in sys.path else None
try:
    from IPython import get_ipython
    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic("load_ext", "autoreload"); ip.run_line_magic("autoreload", "2")
        print("autoreload : enabled")
except Exception as exc:
    print(f"autoreload : skipped ({exc})")
import torch
print(f"torch {torch.__version__}  CUDA available={torch.cuda.is_available()} count={torch.cuda.device_count()}")


autoreload : enabled
torch 2.11.0+cu128  CUDA available=True count=4


## Step 1 - Checkpoints + 100-band PCA (Houston)
Ensures the upstream MAE checkpoints exist and fits `pca_houston_100band.pkl` (CPU) if missing. Trento/MUUFL need no PCA (they resample).


In [4]:
from pathlib import Path
from data.datasets import get_spec
from models.hypersigma.preprocessing import fit_dataset_pca, fit_pca_output_stats, load_pca

for name in ("spat-vit-base.pth", "spec-vit-base.pth"):
    p = Path("checkpoints/hypersigma") / name
    if not p.exists():
        raise FileNotFoundError(
            f"{p} missing. Run: bash scripts/download_hypersigma_checkpoints.sh checkpoints/hypersigma")
    print(f"  [OK] {p} ({p.stat().st_size/1e6:.0f} MB)")

for ds in datasets:
    spec = get_spec(ds)
    if spec.hsi_channels < NATIVE_SPAT_IN_CHANS:
        print(f"  {ds:8s}: {spec.hsi_channels} bands < 100 -> resample at run time (no PCA fit)")
        continue
    pca_path = PCA_DIR / f"pca_{ds}_{NATIVE_SPAT_IN_CHANS}band.pkl"
    stats_path = PCA_DIR / f"pca_{ds}_{NATIVE_SPAT_IN_CHANS}band_stats.pkl"
    if pca_path.exists():
        print(f"  {ds:8s}: found {pca_path}")
        continue
    print(f"  {ds:8s}: fitting PCA 144->100 (CPU) ...")
    dset = spec.patched_cls(data_root="./data/raw", patch_size=11, split="all", normalize=True)
    pca = fit_dataset_pca(dataset=dset, n_components=NATIVE_SPAT_IN_CHANS, save_path=str(pca_path))
    fit_pca_output_stats(dataset=dset, pca=pca, save_path=str(stats_path))
    print(f"  {ds:8s}: saved {pca_path} (+stats)")


  [OK] checkpoints/hypersigma/spat-vit-base.pth (1436 MB)
  [OK] checkpoints/hypersigma/spec-vit-base.pth (1392 MB)
  houston : found checkpoints/hypersigma/pca_houston_100band.pkl
  trento  : 63 bands < 100 -> resample at run time (no PCA fit)
  muufl   : 64 bands < 100 -> resample at run time (no PCA fit)


/work/nmaric/CoFFE/CoFFE/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## Step 2 - Bootstrap the umbrella experiment dir


In [5]:
from lib.experiments import ExperimentLogger
logger_ = ExperimentLogger(repo_root=REPO)
try:
    logger_.get_experiment(experiment_name); print(f"Experiment '{experiment_name}' exists")
except FileNotFoundError:
    logger_.start_pretrain(name=experiment_name,
        description="HyperSIGMA native-geometry MAE SEM-tuning (sem_only): frozen native encoder, "
                    "fused features tuned via masked reconstruction, evaluated few-shot.",
        config={"meta": "native-geometry SEM-tuning ablation"})
    print(f"Created experiment '{experiment_name}'")


Created experiment 'hypersigma_native_sem_run1'


## Step 3 - Adapt (sem_only, native) then evaluate (fused) per dataset
For each dataset: deep-merge per-dataset overrides onto the Houston native-SEM config, run the SEM-tuning adaptation, then few-shot eval the adapted **fused** features at native geometry. Check each adapt `pretrain.log` for `Native-geometry ablation` + that only SEM/fused/l1/mask-token params are trainable, and each eval `eval.log` for `mode=fused`, `native_geometry`, and an adapted load with `missing=0 / unexpected=0`.


In [ ]:
import json
from pathlib import Path
from lib.adapt_runner import run_adapt_hypersigma
from lib.eval_runner import run_hypersigma_evaluation

runs = []
for ds in datasets:
    spec = get_spec(ds)
    adapt_name = f"hypersigma_adapt_{ds}_native_sem_run1"
    overrides = {
        "model": {"hsi_channels": spec.hsi_channels},
        "data": {"dataset": ds},
        "paths": {
            "native_pca_spat_path": native_pca_path(ds),   # None -> resample (trento/muufl)
            "checkpoint_dir": f"checkpoints/hypersigma_adapted/{ds}_native_sem",
            "log_dir": f"logs/hypersigma_adapted/{ds}_native_sem",
        },
        "pretrain": {"epochs": adapt_epochs},
        "hardware": {"device": adapt_device},
    }
    print(f"\n{'='*70}\n>>> ADAPT {ds} (sem_only, native, {input_fit})\n{'='*70}")
    exp = run_adapt_hypersigma(
        name=adapt_name,
        description=f"Native-geometry SEM-tuning (sem_only) on {ds}; frozen pretrained encoder, "
                    f"input_fit={input_fit}, spatial 100ch.",
        config=adapt_config_path, overrides=overrides, overwrite=overwrite,
    )
    ckpt = Path(exp.checkpoints_dir) / "checkpoint_final.pth"
    if not ckpt.exists():
        raise FileNotFoundError(f"adapted checkpoint not found: {ckpt}")

    print(f"\n>>> EVAL {ds} (fused, native, adapted)")
    eval_params = {
        "dataset": ds, "mode": "fused",
        "native_geometry": True, "input_fit": input_fit,
        "k_shot": k_shot, "k_query": k_query, "num_episodes": num_episodes,
        "split": "all", "seed": 42, "device": eval_device,
        "distance_metric": "euclidean", "temperature": 10.0,
        "prototype_mode": "mean_features",
        "num_example_episodes": 1, "max_tsne_samples": 100, "no_plots": False,
    }
    ev = run_hypersigma_evaluation(
        experiment_name=experiment_name, eval_name=f"native_sem_{ds}",
        adapted_checkpoint=str(ckpt), eval_params=eval_params, overwrite=overwrite,
    )
    res = json.loads(ev.results_path.read_text())
    runs.append({"dataset": ds, "adapt": adapt_name, "ckpt": str(ckpt),
                 "eval_root": str(ev.root), "results": res})
    print(f"    done -> {ev.root}")
print(f"\nCompleted {len(runs)} adapt+eval runs.")


## Step 4 - Summary (fused OA/AA/Kappa)


In [ ]:
primary = "euclidean"
print(f"Native-geometry SEM-tuning - fused {primary} OA / AA / Kappa (mean +/- CI95)\n")
print(f"{'dataset':8s} {'OA':>16s} {'AA':>16s} {'Kappa':>16s}")
print("-"*56)
for r in runs:
    blk = r["results"].get(primary, {})
    oa, aa, kp = blk.get("OA", {}), blk.get("AA", {}), blk.get("Kappa", {})
    print(f"{r['dataset']:8s} "
          f"{oa.get('mean', float('nan')):>7.2f} +/-{oa.get('ci_95', 0):<5.2f} "
          f"{aa.get('mean', float('nan')):>7.2f} +/-{aa.get('ci_95', 0):<5.2f} "
          f"{kp.get('mean', float('nan')):>7.2f} +/-{kp.get('ci_95', 0):<5.2f}")
